# TD SSVEP — faire jouer un EEG enregistré

Bienvenue. Ce notebook est volontairement guidé.

> **Avant de commencer :** gardez [`README.md`](README.md) ouvert et lancez une fois `python setup_students.py` dans le terminal VS Code.
>
> Les données EEG sont déjà incluses dans le dossier distribué : aucun téléchargement de dataset n'est prévu pendant la séance.
>
> **Pendant le TD :** vous ne modifiez que les cellules marquées **✏️ À VOUS**.


In [ ]:
import sys
print('Python du notebook :', sys.executable)
print('✓ Utilisez le même Python que celui indiqué par setup_students.py')


## 1 — Vérifier que tout est prêt

Cette cellule charge les fonctions déjà écrites pour vous et vérifie que les données du TD sont présentes.

In [ ]:
from _engine.student_api import (
    load_classroom_data, show_posterior_eeg, compute_tune_scores,
    plot_score_distributions, evaluate_windows, compare_settings,
    print_settings_table, evaluate_dataset, evaluate_controller, play_dataset
)

tune = load_classroom_data("tune")
print("✓ Données chargées")
print("  fréquence d'échantillonnage :", tune.sfreq, "Hz")
print("  canaux :", ", ".join(tune.channels))
print("  dataset :", tune.meta.get("dataset"))

## 2 — Regarder un vrai EEG

Le casque a enregistré des tensions au cours du temps. Il n'a pas enregistré directement des « pensées » ni des commandes JUMP.

In [ ]:
show_posterior_eeg(tune)

## 3 — Baseline commune

Nous commençons tous avec **O1/O2**, deux capteurs postérieurs cohérents avec un paradigme visuel.

La fréquence cible du TD est 12 Hz.

In [ ]:
# Baseline commune — ne changez pas encore ces valeurs
channels = ["O1", "O2"]
target_freq = 12.0
harmonics = 2
step_s = 0.25

## 4 — ✏️ À VOUS : choisir une première fenêtre

`window_s` est la quantité de passé EEG utilisée pour produire un score.

Commencez à 2 secondes, puis vous pourrez tester 1 / 1.5 / 2 / 3 secondes.

In [ ]:
# ✏️ À VOUS
window_s = 2.0

times, scores, states = compute_tune_scores(
    tune, channels, window_s, step_s, target_freq, harmonics
)
print("Fenêtre :", window_s, "s")
print("Nombre de fenêtres exploitables :", int((states >= 0).sum()))

## 5 — Le score CCA sépare-t-il JUMP et NONE ?

Si le score tend à être plus élevé pendant JUMP, nous pouvons essayer de placer un seuil entre les deux distributions.

Les fenêtres ambiguës au moment des transitions sont automatiquement exclues de cette figure.

In [ ]:
plot_score_distributions(states, scores)

## 6 — ✏️ À VOUS : choisir un seuil

Le seuil transforme un **score** en **décision**.

- trop bas : beaucoup de JUMP, y compris de faux JUMP ;
- trop haut : système prudent, mais beaucoup de NONE.

In [ ]:
# ✏️ À VOUS
threshold = 0.45

metrics = evaluate_windows(states, scores, threshold)
print('Seuil :', threshold)
print(f"  hit rate (fenêtres JUMP)      : {metrics['hit_rate']:.2f}")
print(f"  false positive rate (fenêtres): {metrics['false_positive_rate']:.2f}")
print(f"  balanced accuracy             : {metrics['balanced_accuracy']:.2f}")


## 7 — Explorer le compromis fenêtre / seuil

Cette cellule compare quelques combinaisons possibles sur les **données de réglage uniquement**.

Les métriques de ce tableau sont des métriques **par fenêtre EEG** : elles ne comptent pas encore des obstacles ou des blocs entiers.

- `hit rate` = proportion de fenêtres JUMP reconnues ;
- `false positive rate` = proportion de fenêtres NONE transformées à tort en JUMP ;
- `balanced accuracy` = moyenne du taux de détection JUMP et du taux de reconnaissance NONE.

Ne cherchez pas mécaniquement le plus gros nombre : choisissez un compromis que vous pouvez expliquer.


In [ ]:
rows = compare_settings(tune, channels)
print_settings_table(rows)

## 8 — ✏️ À VOUS : choix final sur les données de réglage

Revenez si nécessaire aux deux cellules **✏️ À VOUS** précédentes.

Quand votre groupe est satisfait, notez ici votre choix final.

### Important — explorer puis verrouiller

`window_s` et `threshold` sont vos paramètres **en cours d'essai**.
La cellule de verrouillage ci-dessous les copie dans `FINAL_WINDOW_S` et `FINAL_THRESHOLD`.

Après ce verrouillage, le test held-out et le jeu utilisent uniquement les variables `FINAL_...`.
Si vous revenez modifier `window_s` ou `threshold`, ré-exécutez ensuite la cellule de verrouillage.


# 🔒 LOCK PARAMETERS

Exécutez cette cellule **une fois votre choix fait**. Après cela, ne choisissez plus vos paramètres en regardant le test.


In [ ]:
# ✏️ À VOUS — recopiez / verrouillez vos choix finaux
FINAL_WINDOW_S = float(window_s)
FINAL_THRESHOLD = float(threshold)
FINAL_CHANNELS = channels.copy()

print('🔒 PARAMÈTRES VERROUILLÉS')
print('  canaux :', FINAL_CHANNELS)
print('  fenêtre :', FINAL_WINDOW_S, 's')
print('  seuil :', FINAL_THRESHOLD)
print('\nSi vous changez window_s ou threshold plus haut, vous devez ré-exécuter CETTE cellule pour que le jeu change.')


## 9 — Test tenu à l'écart : même personne, autre session

In [ ]:
game_data = load_classroom_data('game')

_, _, _, heldout_windows = evaluate_dataset(
    game_data, FINAL_CHANNELS, FINAL_WINDOW_S, FINAL_THRESHOLD,
    target_freq=target_freq, harmonics=harmonics
)
heldout_controller = evaluate_controller(
    game_data, FINAL_CHANNELS, FINAL_WINDOW_S, FINAL_THRESHOLD,
    target_freq=target_freq, harmonics=harmonics,
    confirm=2, cooldown_s=1.0
)

print('HELD-OUT — métriques par fenêtre (session complète)')
print(f"  hit rate            : {heldout_windows['hit_rate']:.2f}")
print(f"  false positive rate : {heldout_windows['false_positive_rate']:.3f}")
print(f"  balanced accuracy   : {heldout_windows['balanced_accuracy']:.2f}")

print('\nHELD-OUT — contrôleur / événements (session complète)')
print(f"  cibles détectées    : {heldout_controller['detected_blocks']}/{heldout_controller['target_blocks']}")
print(f"  faux JUMP           : {heldout_controller['false_jumps']}")
print(f"  actions             : {heldout_controller['actions']}")
lat = heldout_controller['median_latency_s']
print('  latence médiane     :', '—' if lat != lat else f'{lat:.2f} s')


## 10 — Lancer le replay gamifié

Le jeu ne rejoue **pas les cinq minutes complètes** de la session : il sélectionne automatiquement de courts extraits autour des 5 blocs 12 Hz.

- le décodage reste causal : chaque score a bien été calculé avec le passé EEG disponible à cet instant ;
- les longs passages sans événement sont simplement sautés pour éviter une démo vide et interminable ;
- le replay est accéléré **×3**, donc la partie dure typiquement 15–25 secondes ;
- **Échap** ou **Q** ferme immédiatement la fenêtre.

La barre violette montre le score CCA ; le trait vert montre votre seuil.


## Même parcours pour tous les groupes

Tous les groupes vont maintenant utiliser **le même EEG held-out et les mêmes 5 obstacles**.

Ce qui change est votre pipeline verrouillé : `FINAL_CHANNELS`, `FINAL_WINDOW_S` et `FINAL_THRESHOLD`.

> **Le but n’est pas d’avoir un jeu aléatoire : le terrain est standardisé pour que vos choix de traitement soient comparables.**

Les annotations du dataset servent à placer les obstacles et calculer les métriques ; elles ne sont jamais fournies au CCA.


## 🎮 Replay gamifié

Le jeu est une **visualisation compacte**, pas une seconde évaluation indépendante.

- les décisions du contrôleur sont calculées **une seule fois par score CCA**, au pas de 0,25 s ;
- la vitesse d'affichage ×3 ne change donc ni les décisions ni les actions ;
- un obstacle est considéré franchi si le contrôleur a produit un JUMP valide pendant son bloc cible ;
- les faux JUMP affichés par le jeu concernent seulement les extraits compacts montrés à l'écran.

Pour juger scientifiquement le pipeline sur toute la session, utilisez les métriques `HELD-OUT` juste au-dessus.


In [ ]:
from _engine.student_api import load_classroom_data, play_dataset

game_data = globals().get('game_data', None)
if game_data is None:
    game_data = load_classroom_data('game')

if 'FINAL_WINDOW_S' in globals() and 'FINAL_THRESHOLD' in globals():
    game_channels = globals().get('FINAL_CHANNELS', ['O1', 'O2'])
    game_window_s = FINAL_WINDOW_S
    game_threshold = FINAL_THRESHOLD
    print('✓ Le jeu utilise les paramètres VERROUILLÉS :')
    print('  window =', game_window_s, 's')
    print('  threshold =', game_threshold)
    print('  channels =', game_channels)

    # Important : modifier seulement window_s plus haut ne change pas un paramètre déjà verrouillé.
    if 'window_s' in globals() and float(window_s) != float(FINAL_WINDOW_S):
        print(f'⚠️ window_s vaut maintenant {window_s}s, mais le jeu utilise encore FINAL_WINDOW_S={FINAL_WINDOW_S}s.')
        print('   Ré-exécutez la cellule 🔒 LOCK PARAMETERS si vous voulez appliquer ce nouveau choix.')
    if 'threshold' in globals() and float(threshold) != float(FINAL_THRESHOLD):
        print(f'⚠️ threshold vaut maintenant {threshold}, mais le jeu utilise encore FINAL_THRESHOLD={FINAL_THRESHOLD}.')
        print('   Ré-exécutez la cellule 🔒 LOCK PARAMETERS si vous voulez appliquer ce nouveau choix.')
else:
    print('⚠️ Les cellules de réglage n’ont pas été verrouillées.')
    print('   Valeurs de démonstration utilisées.')
    game_channels = ['O1', 'O2']
    game_window_s = 2.0
    game_threshold = 0.45

game_target_freq = globals().get('target_freq', 12.0)
game_harmonics = globals().get('harmonics', 2)

game_metrics = play_dataset(
    game_data, game_channels, game_window_s, game_threshold,
    target_freq=game_target_freq, harmonics=game_harmonics,
    confirm=2, cooldown_s=1.0, speed=3.0,
    compact=True, max_targets=5
)

print('\nREPLAY COMPACT — visualisation du jeu')
print(f"  paramètres : window={game_metrics['window_s_used']:.2f}s, threshold={game_metrics['threshold_used']:.2f}")
print(f"  cibles détectées dans les extraits : {game_metrics['detected_blocks']}/{game_metrics['target_blocks']}")
print(f"  faux JUMP dans les extraits        : {game_metrics['false_jumps']}")
print(f"  obstacles franchis                 : {game_metrics['game_obstacles_cleared']}")
print(f"  collisions                         : {game_metrics['game_collisions']}")


## 11 — Bonus : nouvelle personne, aucun réentraînement

Nous appliquons exactement les mêmes réglages à un autre participant. C'est une première expérience de généralisation inter-individuelle.

In [ ]:
other = load_classroom_data('generalization')
_, _, _, other_windows = evaluate_dataset(
    other, FINAL_CHANNELS, FINAL_WINDOW_S, FINAL_THRESHOLD,
    target_freq=target_freq, harmonics=harmonics
)
other_controller = evaluate_controller(
    other, FINAL_CHANNELS, FINAL_WINDOW_S, FINAL_THRESHOLD,
    target_freq=target_freq, harmonics=harmonics,
    confirm=2, cooldown_s=1.0
)

print('NOUVEAU SUJET — mêmes paramètres, aucun retuning')
print(f"  balanced accuracy : {other_windows['balanced_accuracy']:.2f}")
print(f"  hit rate          : {other_windows['hit_rate']:.2f}")
print(f"  false positive rate: {other_windows['false_positive_rate']:.3f}")
print(f"  cibles détectées  : {other_controller['detected_blocks']}/{other_controller['target_blocks']}")
print(f"  faux JUMP         : {other_controller['false_jumps']}")


## 12 — Ce que votre groupe doit savoir expliquer

Répondez oralement ou dans vos notes :

1. Pourquoi O1/O2 ?
2. Qu'est-ce que le score CCA mesure intuitivement ?
3. Pourquoi `0.70` n'est-il pas « 70 % de probabilité de JUMP » ?
4. Quel compromis avez-vous fait avec `window_s` ?
5. Quel compromis avez-vous fait avec `threshold` ?
6. Quelle différence entre **score**, **décision** et **commande** ?
7. Pourquoi avons-nous verrouillé les paramètres avant le test ?
8. Que se passe-t-il sur le nouveau participant ?

> **Une baisse de performance n'est pas un échec du TD.** C'est une observation sur la généralisation du système.